## 1. Chuẩn bị Google Colab

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Cài đặt Roboflow

In [ ]:
!pip -q install -U roboflow

## 3. Lưu Roboflow API Key bằng Colab Secrets

In [ ]:
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
print("Roboflow secret loaded:", bool(ROBOFLOW_API_KEY))

## 4. Tải dataset từ Roboflow

In [ ]:
from roboflow import Roboflow

WORKSPACE_ID = "hai-nam-mai"
PROJECT_ID = "sitting-posture-classification-6vwq1-f4y4a"
VERSION = 1

rf = Roboflow(api_key = ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_ID).project(PROJECT_ID)
version = project.version(VERSION)
dataset = version.download(
    model_format = "folder",
    location = "/content/roboflow_dataset"
)
DATA_DIR = dataset.location
print("Dataset location:", DATA_DIR)

## 5. Kiểm tra cấu trúc dữ liệu

In [ ]:
import os
print("DATA_DIR =", DATA_DIR)
print(os.listdir(DATA_DIR))

## 6. Tìm đường dẫn train

In [ ]:
from pathlib import Path
root = Path(DATA_DIR)
def find_split(candidates):
    for name in candidates:
        p = root / name
        if p.exists():
            return str(p)
    return None
TRAIN_DIR = find_split(["train"])
print("TRAIN_DIR:", TRAIN_DIR)
assert TRAIN_DIR is not None, "Không tìm thấy train."

## 7. Load dữ liệu huấn luyện

In [ ]:
IMG_SIZE = (224, 224) # Kích thước ảnh
BATCH_SIZE = 16 # số lượng mẫu dữ liệu được đưa vào mô hình học máy trong 1 lần tính toán
SEED = 42

# Load the full dataset from the TRAIN_DIR
full_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

# Get class names from the full dataset
class_names = full_ds.class_names

# Calculate dataset sizes
DATASET_SIZE = full_ds.cardinality().numpy() * BATCH_SIZE
TRAIN_SIZE = int(0.7 * DATASET_SIZE)
VALID_SIZE = int(0.15 * DATASET_SIZE)
TEST_SIZE = DATASET_SIZE - TRAIN_SIZE - VALID_SIZE

# Split the dataset
train_ds = full_ds.take(TRAIN_SIZE // BATCH_SIZE)
remaining_ds = full_ds.skip(TRAIN_SIZE // BATCH_SIZE)

valid_ds = remaining_ds.take(VALID_SIZE // BATCH_SIZE)
test_ds = remaining_ds.skip(VALID_SIZE // BATCH_SIZE)

print("Classes:", class_names)
print(f"Total images: {DATASET_SIZE}")
print(f"Train images: {train_ds.cardinality().numpy() * BATCH_SIZE}")
print(f"Valid images: {valid_ds.cardinality().numpy() * BATCH_SIZE}")
print(f"Test images: {test_ds.cardinality().numpy() * BATCH_SIZE}")

## 8. Kiểm tra dữ liệu trước khi huấn luyện mô hình

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize = (9, 9))
for images, labels in train_ds.take(1):
    n = min(9, len(images))
    for i in range(n):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Prefetch dữ liệu

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
valid_ds = valid_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 10. Học chuyển đổi (Transfer Learning) sử dụng MobileNetV2v

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False
inputs = keras.Input(shape=(224, 224, 3))
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(
    len(class_names),
    activation="softmax"
)(x)
model = keras.Model(inputs, outputs)
model.summary()

## 11. Compile mô hình

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 12. Huấn luyện mô hình

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 152s 978ms/step - accuracy: 0.6952 - loss: 0.8522 - val_accuracy: 0.8468 - val_loss: 0.5078
Epoch 2/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 146s 996ms/step - accuracy: 0.8601 - loss: 0.4541 - val_accuracy: 0.8730 - val_loss: 0.3824
Epoch 3/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - accuracy: 0.8882 - loss: 0.3749 - val_accuracy: 0.9173 - val_loss: 0.2992
Epoch 4/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 150s 1s/step - accuracy: 0.9107 - loss: 0.3091 - val_accuracy: 0.9335 - val_loss: 0.2407
Epoch 5/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 144s 977ms/step - accuracy: 0.9082 - loss: 0.2727 - val_accuracy: 0.9173 - val_loss: 0.2377
Epoch 6/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 145s 985ms/step - accuracy: 0.9281 - loss: 0.2424 - val_accuracy: 0.9335 - val_loss: 0.2223
Epoch 7/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 145s 987ms/step - accuracy: 0.9298 - loss: 0.2241 - val_accuracy: 0.9496 - val_loss: 0.2049
Epoch 8/10
147/147 ━━━━━━━━━━━━━━━━━━━━ 139s 945ms/step - accuracy: 0.9375 - loss:

## 13. Training curves và Overfitting

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 14. Đánh giá trên tập dữ liệu test

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

## 15. Vẽ confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
y_true = []
y_pred = []

for images, labels in test_ds:
    y_true.extend(labels.numpy())
    predictions = model.predict(images)
    y_pred.extend(tf.argmax(predictions, axis=1).numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("y_true shape:", y_true.shape)
print("y_pred shape:", y_pred.shape)
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm)

ax.set_xticks(np.arange(len(class_names)))
ax.set_yticks(np.arange(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 16. Dự đoán

In [ ]:
from google.colab import files
uploaded_image = files.upload()
image_filename = list(uploaded_image.keys())[0]
print("Uploaded:", image_filename)

In [ ]:
img = tf.keras.utils.load_img(
    image_filename,
    target_size=IMG_SIZE
)

img_array = tf.keras.utils.img_to_array(img)
batch = tf.expand_dims(img_array, axis=0)

pred = model.predict(batch, verbose=0)[0]
pred_index = int(np.argmax(pred))

plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis("off")
plt.title(f"{class_names[pred_index]} | score={pred[pred_index]:.3f}")
plt.show()

print("Prediction:", class_names[pred_index])
print("Model score:", float(pred[pred_index]))
print("\nScores by class:")

for name, score in zip(class_names, pred):
    print(f"- {name}: {float(score):.4f}")

## 17. Lưu lại mô hình

In [ ]:
MODEL_PATH = "/content/image_classifier_v1.keras"
model.save(MODEL_PATH)
print("Saved:", MODEL_PATH)

## 18. Demo giao diện mô hình

In [ ]:
import gradio as gr

def predict_image(image_file):
    if image_file is None:
        return "Vui lòng tải lên một ảnh."

    img = tf.keras.utils.load_img(
        image_file,
        target_size=IMG_SIZE
    )

    img_array = tf.keras.utils.img_to_array(img)
    batch = tf.expand_dims(img_array, axis=0)

    pred = model.predict(batch, verbose=0)[0]
    pred_index = int(np.argmax(pred))

    predicted_class = class_names[pred_index]
    score = float(pred[pred_index])

    return f"Prediction: {predicted_class} | Score: {score:.3f}"


image_input = gr.Image(type="filepath", label="Tải lên ảnh của bạn")
text_output = gr.Textbox(label="Dự đoán của mô hình")

gradio_interface = gr.Interface(
    fn=predict_image,
    inputs=image_input,
    outputs=text_output,
    title="Image Classification Predictor",
    description="Tải lên một ảnh để mô hình phân loại và đưa ra dự đoán."
)

gradio_interface.launch(share=True)

In [ ]:
from google.colab import files
files.download("/content/image_classifier_v1.keras")